# Sync vs Async in Python — A Real-World Example

Imagine you're making **breakfast** with three items:

| Dish    | Time it takes |
| ------- | ------------- |
| Coffee  | 3 seconds     |
| Toast   | 2 seconds     |
| Eggs    | 4 seconds     |

The machines (kettle, toaster, pan) do the work — you're just **waiting** for them.

- **Synchronous way:** start coffee → wait → start toast → wait → start eggs → wait. Total = 3 + 2 + 4 = **9 seconds**.
- **Asynchronous way:** start all three at once, wait together. Total = the slowest one = **4 seconds**.

The cooking itself isn't faster — async just stops you from standing idle while each machine does its thing. This is exactly what happens when your code waits on APIs, databases, or file downloads.

## Installation

Nothing to install — `asyncio` is built into Python.

## The Synchronous Version

We use `time.sleep(...)` to simulate cooking time. Each call blocks the program until it finishes.

In [1]:
import time

def make_coffee():
    print("Starting coffee")
    time.sleep(3)
    print("Coffee ready")
    return "coffee"

def make_toast():
    print("Starting toast")
    time.sleep(2)
    print("Toast ready")
    return "toast"

def make_eggs():
    print("Starting eggs")
    time.sleep(4)
    print("Eggs ready")
    return "eggs"

start = time.perf_counter()

make_coffee()
make_toast()
make_eggs()

print(f"\nTotal time: {time.perf_counter() - start:.2f}s")

Starting coffee
Coffee ready
Starting toast
Toast ready
Starting eggs
Eggs ready

Total time: 9.00s


Expected output: **~9 seconds** — each dish has to finish before the next one starts.

## The Asynchronous Version

Same three dishes, but:

- `async def` makes each function a **coroutine** (a task that can be paused).
- `await asyncio.sleep(...)` says *"pause me here, let other tasks run while I wait"*.
- `asyncio.gather(...)` starts all three coroutines **at the same time** and waits for them all to finish.

In [2]:
import asyncio, time

async def make_coffee():
    print("Starting coffee")
    await asyncio.sleep(3)
    print("Coffee ready")
    return "coffee"

async def make_toast():
    print("Starting toast")
    await asyncio.sleep(2)
    print("Toast ready")
    return "toast"

async def make_eggs():
    print("Starting eggs")
    await asyncio.sleep(4)
    print("Eggs ready")
    return "eggs"

start = time.perf_counter()

results = await asyncio.gather(
    make_coffee(),
    make_toast(),
    make_eggs(),
)

print("\nServed:", results)
print(f"Total time: {time.perf_counter() - start:.2f}s")

Starting coffee
Starting toast
Starting eggs
Toast ready
Coffee ready
Eggs ready

Served: ['coffee', 'toast', 'eggs']
Total time: 4.01s


Expected output: **~4 seconds** — all three dishes cook in parallel, so you only wait as long as the slowest one (eggs).

Notice the print order too: in the sync version each "Starting" is followed by its own "ready". In the async version all three "Starting" lines appear immediately, then the "ready" lines arrive in the order each dish finishes (toast → coffee → eggs).

## Takeaway

| | Sync | Async |
|---|---|---|
| Tasks run | one after another | all at the same time |
| Total time | sum of all tasks (3+2+4 = 9s) | time of the slowest task (4s) |
| Best for | quick or CPU work | waiting on I/O (APIs, DB, files) |

Use async whenever your program would otherwise just be **waiting** — that's where the speed-up comes from.

## Running This in a `.py` Script

In a Jupyter notebook you can write `await ...` directly in a cell because Jupyter already has an event loop running.

In a normal Python script there is **no** event loop, so `await` at the top level gives a `SyntaxError`. You need to:

1. Wrap your async code in an `async def main()` function.
2. Start the event loop **once** with `asyncio.run(main())`.

```python
import asyncio

async def main():
    await asyncio.gather(
        make_coffee(),
        make_toast(),
        make_eggs(),
    )

if __name__ == "__main__":
    asyncio.run(main())   # only ONE asyncio.run() per program
```

**Rules of thumb:**
- `asyncio.run()` is the entry point from sync → async. Call it once, at the top of your script.
- Inside any `async def`, use `await` (never call `asyncio.run` again — it creates a new loop and will error).
- In Jupyter, skip `asyncio.run()` entirely and just use `await`.